# Package Model
Steps for packing a model from a training job and making it available for inference tasks such as an endpoint or transform job.

Once created, models can be viewed in the SageMaker Console.
Navigate to the "Inference" section on the left sidebar, and go to "Models".
The new model with your `model_name` will appear in the list.

For purposes of this notebook, we are assuming the use of a PyTorch models.

## Model Information
Fill out the section below with the desired values

In [ ]:
# Name of SageMaker training job
training_job_name = "test-count-8-13446-it0-240419-1930-2"
training_job_name = "17k-dataset-cleaned-1--1-it0-241018-1441"

# What to name the Model.
# NOTE: Reusing a model name will OVERWRITE the existing model
# Please ensure a unique model name or that you want to overwrite the existing model
model_name = "test-count-8-13446-it0-240419-1930-2"
model_name = "17k-dataset-cleaned-1--1-it0-241018-1441"

# Name of the entrypoint file for inference
# See this documentation for details of what the inference script should include:
# https://sagemaker.readthedocs.io/en/stable/frameworks/pytorch/using_pytorch.html#serve-a-pytorch-model
inference_entrypoint = "inference.py"
inference_entrypoint = "batch_inference_cleaned_100smpls.py"

# OPTIONAL VARIABLES
# Source directory to include with the model. By default, will re-use the source code
# used during training. But, we can optionally provide a new directory. For example,
# if updating the inference script behavior for an existing model.
src_dir = r"../src/"

# Update if you want to deploy on a GPU enabled device. This lets the model packaging code
# know whether to use a CPU or GPU optimized inference container.
deploy_instance_type = "ml.g4dn.4xlarge"

# If you need to attach a custom role for the deployed model service, place the Role ARN here.
# Otherwise, will use the default SM Role
role = None

## Package the Model
Do not edit the code below unless required for specific deployments

In [ ]:
from sagemaker import get_execution_role
from sagemaker.pytorch import PyTorch
import os

In [ ]:
max_payload_size_mb=25
workers_per_model = 4
env_variables = {
    # Update the maximum payload size for TS. Needs to match SageMaker's max payload size
    # Default for TS is 6,553,500 bytes
    # Convert max_payload_size in MB to Bytes
    "TS_DEFAULT_WORKERS_PER_MODEL": str(workers_per_model),
    "TS_MAX_REQUEST_SIZE": str(max_payload_size_mb * 1024 * 1024)
}

# Get the execution role arn if not provided above
role = os.getenv('AWS_ROLE') #role or get_execution_role()

In [ ]:
# Create the Model
training_estimator = PyTorch.attach(training_job_name)

model_package = training_estimator.create_model(
    name=model_name,
    entry_point=inference_entrypoint,
    source_dir=src_dir,
    role=role,
    env=env_variables,
    model_server_workers=workers_per_model,
)

model_package.create(deploy_instance_type)